In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
training_data= pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv") 
testing_data= pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")

In [4]:
training_data.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [5]:
training_data.shape

(630000, 21)

In [6]:
testing_data.shape

(270000, 20)

In [7]:
training_data.isnull().sum()

id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64

In [8]:
training_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Soil_Type                630000 non-null  object 
 2   Soil_pH                  630000 non-null  float64
 3   Soil_Moisture            630000 non-null  float64
 4   Organic_Carbon           630000 non-null  float64
 5   Electrical_Conductivity  630000 non-null  float64
 6   Temperature_C            630000 non-null  float64
 7   Humidity                 630000 non-null  float64
 8   Rainfall_mm              630000 non-null  float64
 9   Sunlight_Hours           630000 non-null  float64
 10  Wind_Speed_kmh           630000 non-null  float64
 11  Crop_Type                630000 non-null  object 
 12  Crop_Growth_Stage        630000 non-null  object 
 13  Season                   630000 non-null  object 
 14  Irri

In [9]:
training_data= training_data.drop_duplicates().reset_index(drop=True)

In [10]:
corr= training_data.corr(numeric_only=True)

fig= px.imshow(corr,text_auto='.2f',aspect="auto")
fig.show()

CREATING new numerical features

In [11]:
training_data['temp_humidity_ratio'] = training_data['Temperature_C'] / (training_data['Humidity'] + 1e-5)
training_data['rainfall_temp_ratio'] = training_data['Rainfall_mm'] / (training_data['Temperature_C'] + 1e-5)
training_data['heat_index_proxy'] = training_data['Temperature_C'] * training_data['Sunlight_Hours']
training_data['wind_chill_proxy'] = training_data['Temperature_C'] * training_data['Wind_Speed_kmh']
    
# Soil interactions
training_data['soil_health_idx'] = training_data['Organic_Carbon'] * training_data['Soil_Moisture'] / (training_data['Soil_pH'] + 1e-5)
training_data['moisture_conductivity'] = training_data['Soil_Moisture'] * training_data['Electrical_Conductivity']
training_data['air_soil_moisture_mismatch'] = training_data['Humidity'] / (training_data['Soil_Moisture'] + 1e-5)
training_data['ph_conductivity_ratio'] = training_data['Soil_pH'] / (training_data['Electrical_Conductivity'] + 1e-5)
    
# Area and Volume interactions
training_data['rain_volume'] = training_data['Rainfall_mm'] * training_data['Field_Area_hectare']
training_data['previous_irrigation_volume'] = training_data['Previous_Irrigation_mm'] * training_data['Field_Area_hectare']
training_data['irrigation_diff'] = training_data['Rainfall_mm'] - training_data['Previous_Irrigation_mm']
training_data['total_water_supply'] = training_data['Rainfall_mm'] + training_data['Previous_Irrigation_mm']
    
# Log transforms for highly skewed data limits outliers effect
training_data['log_Rainfall_mm'] = np.log1p(training_data['Rainfall_mm'])
training_data['log_Previous_Irrigation_mm'] = np.log1p(training_data['Previous_Irrigation_mm'])

#ranking features
training_data['Soil_Moisture_rank']= training_data['Soil_Moisture'].rank(pct=True)
training_data['EC_rank']= training_data['Electrical_Conductivity'].rank(pct=True)
training_data['Rainfall_rank']= training_data['Rainfall_mm'].rank(pct=True)
training_data['Irrigation_rank']= training_data['Previous_Irrigation_mm'].rank(pct=True)

training_data['water_balance']= training_data['Rainfall_mm']- training_data['Soil_Moisture']
training_data['Soil_health']= training_data['Soil_pH']*training_data['Electrical_Conductivity']
training_data['Soil_moisture_sqrt']= np.sqrt(training_data['Soil_Moisture'])

#training_data['moisture_to_rain_ratio'] = training_data['Soil_Moisture'] / (training_data['Rainfall_mm'] + 1e-5)
#training_data['irrigation_efficiency'] = training_data['Soil_Moisture'] / (training_data['Previous_Irrigation_mm'] + 1e-5)
#training_data['temp_x_rain'] = training_data['Temperature_C'] * training_data['Rainfall_mm']
#training_data['temp_humidity_diff'] = training_data['Temperature_C'] - training_data['Humidity']

#non linear features
#training_data['temp_squared'] = training_data['Temperature_C'] ** 2
#training_data['temp_cubed'] = training_data['Temperature_C'] ** 3
#training_data['humidity_power'] = training_data['Humidity'] ** 1.5
#training_data['soil_moisture_log_sq'] = np.log1p(training_data['Soil_Moisture']) ** 2
#training_data['rain_sqrt_temp'] = np.sqrt(training_data['Rainfall_mm']) * training_data['Temperature_C']
#training_data['ec_power'] = training_data['Electrical_Conductivity'] ** 1.3
#training_data['ph_squared'] = training_data['Soil_pH'] ** 2
#training_data['temp_moisture_nl'] = (training_data['Temperature_C'] ** 2) * np.sqrt(training_data['Soil_Moisture'])
#training_data['log_rain_temp'] = np.log1p(training_data['Rainfall_mm']) * training_data['Temperature_C']

In [12]:
testing_data['temp_humidity_ratio'] = testing_data['Temperature_C'] / (testing_data['Humidity'] + 1e-5)
testing_data['rainfall_temp_ratio'] = testing_data['Rainfall_mm'] / (testing_data['Temperature_C'] + 1e-5)
testing_data['heat_index_proxy'] = testing_data['Temperature_C'] * testing_data['Sunlight_Hours']
testing_data['wind_chill_proxy'] = testing_data['Temperature_C'] * testing_data['Wind_Speed_kmh']
    
# Soil interactions
testing_data['soil_health_idx'] = testing_data['Organic_Carbon'] * testing_data['Soil_Moisture'] / (testing_data['Soil_pH'] + 1e-5)
testing_data['moisture_conductivity'] = testing_data['Soil_Moisture'] * testing_data['Electrical_Conductivity']
testing_data['air_soil_moisture_mismatch'] = testing_data['Humidity'] / (testing_data['Soil_Moisture'] + 1e-5)
testing_data['ph_conductivity_ratio'] = testing_data['Soil_pH'] / (testing_data['Electrical_Conductivity'] + 1e-5)
    
# Area and Volume interactions
testing_data['rain_volume'] = testing_data['Rainfall_mm'] * testing_data['Field_Area_hectare']
testing_data['previous_irrigation_volume'] = testing_data['Previous_Irrigation_mm'] * testing_data['Field_Area_hectare']
testing_data['irrigation_diff'] = testing_data['Rainfall_mm'] - testing_data['Previous_Irrigation_mm']
testing_data['total_water_supply'] = testing_data['Rainfall_mm'] + testing_data['Previous_Irrigation_mm']
    
# Log transforms for highly skewed data limits outliers effect
testing_data['log_Rainfall_mm'] = np.log1p(testing_data['Rainfall_mm'])
testing_data['log_Previous_Irrigation_mm'] = np.log1p(testing_data['Previous_Irrigation_mm'])

#ranking features
testing_data['Soil_Moisture_rank']= testing_data['Soil_Moisture'].rank(pct=True)
testing_data['EC_rank']= testing_data['Electrical_Conductivity'].rank(pct=True)
testing_data['Rainfall_rank']= testing_data['Rainfall_mm'].rank(pct=True)
testing_data['Irrigation_rank']= testing_data['Previous_Irrigation_mm'].rank(pct=True)

testing_data['water_balance']= testing_data['Rainfall_mm']- testing_data['Soil_Moisture']
testing_data['Soil_health']= testing_data['Soil_pH']*testing_data['Electrical_Conductivity']
testing_data['Soil_moisture_sqrt']= np.sqrt(testing_data['Soil_Moisture'])

#testing_data['moisture_to_rain_ratio'] = testing_data['Soil_Moisture'] / (testing_data['Rainfall_mm'] + 1e-5)
#testing_data['irrigation_efficiency'] = testing_data['Soil_Moisture'] / (testing_data['Previous_Irrigation_mm'] + 1e-5)
#testing_data['temp_x_rain'] = testing_data['Temperature_C'] * testing_data['Rainfall_mm']
#testing_data['temp_humidity_diff'] = testing_data['Temperature_C'] - testing_data['Humidity']

#non linear features
#testing_data['temp_squared'] = testing_data['Temperature_C'] ** 2
#testing_data['temp_cubed'] = testing_data['Temperature_C'] ** 3
#testing_data['humidity_power'] = testing_data['Humidity'] ** 1.5
#testing_data['soil_moisture_log_sq'] = np.log1p(testing_data['Soil_Moisture']) ** 2
#testing_data['rain_sqrt_temp'] = np.sqrt(testing_data['Rainfall_mm']) * testing_data['Temperature_C']
#testing_data['ec_power'] = testing_data['Electrical_Conductivity'] ** 1.3
#testing_data['ph_squared'] = testing_data['Soil_pH'] ** 2
#testing_data['temp_moisture_nl'] = (testing_data['Temperature_C'] ** 2) * np.sqrt(testing_data['Soil_Moisture'])
#testing_data['log_rain_temp'] = np.log1p(testing_data['Rainfall_mm']) * testing_data['Temperature_C']

creating new categorical features

In [13]:
#training_data['Crop_Soil_Combo'] = training_data['Crop_Type'].astype(str) + "_" + training_data['Soil_Type'].astype(str)
#training_data['Region_Season_Combo'] = training_data['Region'].astype(str) + "_" + training_data['Season'].astype(str)

In [14]:
#testing_data['Crop_Soil_Combo'] = testing_data['Crop_Type'].astype(str) + "_" + testing_data['Soil_Type'].astype(str)
#testing_data['Region_Season_Combo'] = testing_data['Region'].astype(str) + "_" + testing_data['Season'].astype(str)

In [15]:
training_data.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,total_water_supply,log_Rainfall_mm,log_Previous_Irrigation_mm,Soil_Moisture_rank,EC_rank,Rainfall_rank,Irrigation_rank,water_balance,Soil_health,Soil_moisture_sqrt
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,838.15,6.588913,4.728803,0.407815,0.895710,0.135232,0.938521,693.41,15.0060,5.707889
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,1032.82,6.894326,3.874529,0.847590,0.579534,0.261962,0.375976,929.05,14.1600,7.523962
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,2312.08,7.697439,4.712948,0.327463,0.838748,0.859519,0.910461,2173.99,16.1027,5.264029
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,1411.18,7.214011,4.004602,0.090727,0.228202,0.440427,0.439052,1344.01,4.9155,3.649658
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,1631.39,7.339018,4.545314,0.891187,0.261221,0.533706,0.755147,1479.06,7.6416,7.690254


correlational analysis after creating new features

In [16]:
corr_2= training_data.corr(numeric_only=True)

fig_2= px.imshow(corr_2,text_auto='.2f',aspect="auto")
fig_2.show()

In [17]:
x= training_data.drop(columns=['Irrigation_Need'])
y= training_data['Irrigation_Need']

In [18]:
print(x[0:5])
print(y[0:5])

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  total_water_supply log_Rainfall_mm  \
0            5.90  ...              838.15        6.588913   
1            6.98  ...             1032.82        6.894326   
2            6.05  ...             2312.08        

encoding targets from low,modium,high to 0,1,2

In [19]:
mapping= {'Low':0,"Medium":1,'High':2}
y= y.map(mapping)

In [20]:
print(np.unique(y))

[0 1 2]


In [21]:
arr,count= np.unique(y,return_counts=True)

In [22]:
print(arr)
print(count)

[0 1 2]
[369917 239074  21009]


There is a data imbalance 

In [23]:
drop_col= ['id']

In [24]:
x= x.drop(columns=drop_col)

In [25]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Soil_Type                   630000 non-null  object 
 1   Soil_pH                     630000 non-null  float64
 2   Soil_Moisture               630000 non-null  float64
 3   Organic_Carbon              630000 non-null  float64
 4   Electrical_Conductivity     630000 non-null  float64
 5   Temperature_C               630000 non-null  float64
 6   Humidity                    630000 non-null  float64
 7   Rainfall_mm                 630000 non-null  float64
 8   Sunlight_Hours              630000 non-null  float64
 9   Wind_Speed_kmh              630000 non-null  float64
 10  Crop_Type                   630000 non-null  object 
 11  Crop_Growth_Stage           630000 non-null  object 
 12  Season                      630000 non-null  object 
 13  Irrigation_Typ

In [26]:
numeric_x= x.select_dtypes(include=['float64'])
category_x= x.select_dtypes(include=['object'])

In [27]:
drop_col_2=['Organic_Carbon','EC_rank','Electrical_Conductivity','heat_index_proxy','Soil_pH','Sunlight_Hours','Field_Area_hectare','soil_health_idx','rain_volume','moisture_conductivity','ph_conductivity_ratio','previous_irrigation_volume','Soil_health','temp_humidity_ratio','total_water_supply']

In [28]:
numeric_x= numeric_x.drop(columns=drop_col_2)

In [29]:
numeric_x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 17 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Soil_Moisture               630000 non-null  float64
 1   Temperature_C               630000 non-null  float64
 2   Humidity                    630000 non-null  float64
 3   Rainfall_mm                 630000 non-null  float64
 4   Wind_Speed_kmh              630000 non-null  float64
 5   Previous_Irrigation_mm      630000 non-null  float64
 6   rainfall_temp_ratio         630000 non-null  float64
 7   wind_chill_proxy            630000 non-null  float64
 8   air_soil_moisture_mismatch  630000 non-null  float64
 9   irrigation_diff             630000 non-null  float64
 10  log_Rainfall_mm             630000 non-null  float64
 11  log_Previous_Irrigation_mm  630000 non-null  float64
 12  Soil_Moisture_rank          630000 non-null  float64
 13  Rainfall_rank 

In [30]:
#category_x= category_x.drop(columns=['Crop_Soil_Combo','Region_Season_Combo'])

In [31]:
#category_x_2= x[['Crop_Soil_Combo','Region_Season_Combo']]

In [32]:
category_x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Soil_Type          630000 non-null  object
 1   Crop_Type          630000 non-null  object
 2   Crop_Growth_Stage  630000 non-null  object
 3   Season             630000 non-null  object
 4   Irrigation_Type    630000 non-null  object
 5   Water_Source       630000 non-null  object
 6   Mulching_Used      630000 non-null  object
 7   Region             630000 non-null  object
dtypes: object(8)
memory usage: 38.5+ MB


In [33]:
#category_x_2.info()

Scaling numeric features

In [34]:
scale= StandardScaler()
numeric_x= scale.fit_transform(numeric_x)

In [35]:
numeric_x[0:3]

array([[-0.2884815 , -1.39015573, -0.55576944, -1.20102851,  1.12745552,
         1.4553669 , -0.36166771, -0.15190302, -0.38542809, -1.28128416,
        -0.85716391,  1.01021131, -0.31934073, -1.26359709,  1.5190775 ,
        -1.19431446, -0.15865275],
       [ 1.17881399, -0.47290687,  0.31950357, -0.77741586, -1.2277795 ,
        -0.44261443, -0.51329922, -1.09765145, -0.59824135, -0.75206902,
        -0.39095329, -0.05179462,  1.20408343, -0.82459093, -0.42963389,
        -0.8095832 ,  1.10423273],
       [-0.58584851, -0.00326618,  1.55554127,  1.20637096, -1.14692814,
         1.40339142,  0.57899188, -0.95585519,  0.67938209,  1.12703836,
         0.83499448,  0.99050092, -0.59768959,  1.24540781,  1.42187644,
         1.22304004, -0.46731051]])

Encoding categorical features

In [36]:
from sklearn.preprocessing import OneHotEncoder

In [37]:
encode= OneHotEncoder(sparse_output=False)

In [38]:
#from category_encoders import TargetEncoder

In [39]:
#te= TargetEncoder(cols=categorical_cols,smoothing=0.3)

In [40]:
#category_x_2= te.fit_transform(category_x_2,y)

In [41]:
#category_x_2= np.asarray(category_x_2)

In [42]:
category_x = encode.fit_transform(category_x)

In [43]:
category_x[0:3]

array([[0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0.,
        1., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0.,
        0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0.]])

x_train_n and x_test_n--->numerical data

In [44]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [45]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,classification_report

Phase 1 is completed where work is done on scaled numeric data only without much feature engineering

In [46]:
print(numeric_x[0])
print(category_x[0])

[-0.2884815  -1.39015573 -0.55576944 -1.20102851  1.12745552  1.4553669
 -0.36166771 -0.15190302 -0.38542809 -1.28128416 -0.85716391  1.01021131
 -0.31934073 -1.26359709  1.5190775  -1.19431446 -0.15865275]
[0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0.
 0. 1. 0. 0. 1. 0. 0. 0.]


In [47]:
print(np.shape(numeric_x))
print(np.shape(category_x))

(630000, 17)
(630000, 32)


In [48]:
params = {
    'max_depth': 5,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'learning_rate': 0.03,
    'n_estimators': 2000,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.5,
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss'
}

In [49]:
#xgb2= XGBClassifier(**params)

In [50]:
#xgb2.fit(numeric_x,y)

In [51]:
#numeric_x_df= pd.DataFrame(numeric_x)

In [52]:
#feat_imp = pd.DataFrame({
 #   'feature': numeric_x_df.columns,
  #  'importance': xgb2.feature_importances_
#}).sort_values(by='importance', ascending=False)

In [53]:
#feat_imp

In [54]:
final_x= np.concatenate((numeric_x,category_x),axis=1)

In [55]:
print(np.shape(final_x))

(630000, 49)


In [56]:
#x_train_f,x_test_f,y_train_f,y_test_f= train_test_split(final_x,y,test_size=0.20,stratify=y,random_state=42)

In [57]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y)
weights = compute_class_weight('balanced', classes=classes, y=y)

class_weights = dict(zip(classes, weights))
class_weights[2] *= 1.22
class_weights[2] *= 1.24
class_weights[2] *= 1.26
class_weights[2] *= 1.28
class_weights[2] *= 1.30
sample_weights = np.array([class_weights[y_i] for y_i in y])

In [58]:
#rf_final= RandomForestClassifier(n_estimators=100,random_state=42)

In [59]:
#rf_final.fit(x_train_f,y_train_f,sample_weight=sample_weights)

In [60]:
#y_pred_f= rf_final.predict(x_test_f)

In [61]:
#accuracy_f= accuracy_score(y_pred_f,y_test_f)
#precision_f= precision_score(y_pred_f,y_test_f,average='weighted')
#recall_f= recall_score(y_pred_f,y_test_f,average='weighted')

In [62]:
#print(accuracy_f*100)
#print(precision_f*100)
#print(recall_f*100)

In [63]:
params = {
    'max_depth': 5,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'learning_rate': 0.03,
    'n_estimators': 2000,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.5,
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss'
}

In [64]:
#xgb= XGBClassifier(**params)

In [65]:
#xgb.fit(final_x,y,sample_weight=sample_weights)

Using averaging in xgboost

In [66]:
seeds = [42, 2024, 99]

models = []

for seed in seeds:
    xgb = XGBClassifier(
        **params,
        random_state=seed
    )
    
    xgb.fit(
        final_x, y,
        sample_weight=sample_weights,
        verbose=0
    )
    
    models.append(xgb)

In [67]:
from lightgbm import LGBMClassifier

lgb= LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=63,
    min_child_sample=50,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb.fit(final_x,y,sample_weight=sample_weights)

[LightGBM] [Warning] Unknown parameter: min_child_sample
[LightGBM] [Warning] Unknown parameter: min_child_sample
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.080528 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4399
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 49
[LightGBM] [Info] Start training from score -1.643220
[LightGBM] [Info] Start training from score -1.643220
[LightGBM] [Info] Start training from score -0.488922


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.02, min_child_sample=50,
               n_estimators=2000, num_leaves=63, random_state=42,
               subsample=0.8)

In [68]:
#categorical_cols=[17,18,19,20,21,22,23,24]

In [69]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.04,
    depth=6,
    loss_function='MultiClass',
    l2_leaf_reg=5,               
    random_strength=1.5,         
    bagging_temperature=0.5,    
    border_count=128,
    early_stopping_rounds=100,
    random_seed=42,
    verbose=0
)

cat.fit(final_x,y)

CatBoostClassifier(bagging_temperature=0.5, border_count=128, depth=6, early_stopping_rounds=100, iterations=1500, l2_leaf_reg=5, learning_rate=0.04, loss_function='MultiClass', random_seed=42, random_strength=1.5, verbose=0)

In [70]:
#y_pred_xgb= xgb.predict(x_test_f)

In [71]:
#print(accuracy_score(y_pred_xgb,y_test_f)*100)
#print(precision_score(y_pred_xgb,y_test_f,average='weighted')*100)
#print(recall_score(y_pred_xgb,y_test_f,average='weighted')*100)

In [72]:
print(np.shape(y))

(630000,)


In [73]:
processed_data= pd.DataFrame(final_x)

In [74]:
processed_data.head()

,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,-0.288482,-1.390156,-0.555769,-1.201029,1.127456,1.455367,-0.361668,-0.151903,-0.385428,-1.281284,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,1.178814,-0.472907,0.319504,-0.777416,-1.227779,-0.442614,-0.513299,-1.097651,-0.598241,-0.752069,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,-0.585849,-0.003266,1.555541,1.206371,-1.146928,1.403391,0.578992,-0.955855,0.679382,1.127038,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,-1.464516,-1.586129,0.000346,-0.171092,-1.417604,-0.247268,1.152029,-1.352263,1.456002,-0.157148,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,1.333298,-0.786001,1.499219,0.123970,0.626529,0.901448,0.421727,0.010039,-0.393123,0.073547,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [75]:
#corr_data= processed_data.corr(numeric_only=True)

In [76]:
#figure= px.imshow(corr_data,text_auto='.2f',aspect='auto')
#figure.show()

In [77]:
testing_data.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,total_water_supply,log_Rainfall_mm,log_Previous_Irrigation_mm,Soil_Moisture_rank,EC_rank,Rainfall_rank,Irrigation_rank,water_balance,Soil_health,Soil_moisture_sqrt
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,...,1580.86,7.335882,3.881151,0.299302,0.833211,0.527406,0.377076,1507.19,17.8716,5.117617
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,...,632.48,6.357929,4.050567,0.029241,0.945361,0.069265,0.456674,566.17,19.1362,3.143247
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,...,565.30,6.303168,3.044522,0.306622,0.220185,0.059169,0.145228,518.75,5.2870,5.152669
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,...,1314.02,7.100052,4.644295,0.795070,0.133822,0.358802,0.841244,1157.45,4.2240,7.319836
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,...,1335.24,7.187589,2.662355,0.887952,0.619674,0.414641,0.079919,1262.89,11.0353,7.682448


In [78]:
testing_x= testing_data.drop(columns=drop_col)

In [79]:
testing_x= testing_x.drop(columns=drop_col_2)

In [80]:
test_x_numeric= testing_x.select_dtypes(include=['float64'])
test_x_category= testing_x.select_dtypes(include=['object'])

In [81]:
#test_x_category= test_x_category.drop(columns=['Crop_Soil_Combo','Region_Season_Combo'])

In [82]:
#test_x_category_2= testing_x[['Crop_Soil_Combo','Region_Season_Combo']]

In [83]:
test_x_numeric= scale.transform(test_x_numeric)

In [84]:
test_x_category= encode.transform(test_x_category)

In [85]:
#est_x_category_2= te.transform(test_x_category_2)

In [86]:
#test_x_category_2= np.asarray(test_x_category_2)

In [87]:
print(np.shape(test_x_numeric))
print(np.shape(test_x_category))
#print(np.shape(test_x_category_2))

(270000, 17)
(270000, 32)


In [88]:
testing_x_combined= np.concatenate((test_x_numeric,test_x_category),axis=1)

In [89]:
xgb_probs = []

for xgb in models:
    p = xgb.predict_proba(testing_x_combined)
    xgb_probs.append(p)

In [90]:
lgb_probs = lgb.predict_proba(testing_x_combined)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



[LightGBM] [Warning] Unknown parameter: min_child_sample


In [91]:
cat_probs = cat.predict_proba(testing_x_combined)

In [92]:
weights = [
    (0.88, 0.10, 0.02),
    (0.89, 0.09, 0.02),
    (0.90, 0.08, 0.02),
    (0.91, 0.07, 0.02),

    (0.88, 0.09, 0.03),
    (0.89, 0.08, 0.03),
    (0.90, 0.07, 0.03),

    (0.87, 0.10, 0.03),
]

In [93]:
bias_grid = [
    (1.0, 1.0, 1.005),
    (1.0, 1.0, 1.008),
    (0.995, 1.0, 1.01),
    (1.0, 0.998, 1.01),
    (0.998, 1.0, 1.012)
]

In [94]:
seed_weights = [
    (0.5, 0.3, 0.2),
    (0.4, 0.4, 0.2),
    (0.6, 0.25, 0.15)
]

In [95]:
p1= xgb_probs[0]
p2= xgb_probs[1]
p3= xgb_probs[2]

In [96]:
#avg_xgb_probs = np.mean(xgb_probs, axis=0)
def temp_scale(probs, T):
    probs = probs ** (1/T)
    return probs / probs.sum(axis=1, keepdims=True)

temps = [0.97, 1.0, 1.03]

avg_lgb_probs= np.mean(lgb_probs,axis=0)
avg_cat_probs= np.mean(cat_probs,axis=0)
for sw in seed_weights:
    avg_xgb_probs=sw[0]*p1 + sw[1]*p2 + sw[2]*p3
    for (w1,w2,w3) in weights:
        base_probs= w1*avg_xgb_probs +w2*avg_lgb_probs +w3*avg_cat_probs
        base_probs /= base_probs.sum(axis=1, keepdims=True)
        for T in temps:
            scaled_probs = temp_scale(base_probs, T)
            for (b0, b1, b2) in bias_grid:
                temp = scaled_probs.copy()
            
                temp[:, 0] *= b0
                temp[:, 1] *= b1
                temp[:, 2] *= b2
                y_pred_test = np.argmax(temp, axis=1)

In [97]:
#y_pred_test= xgb.predict(testing_x_combined)

In [98]:
predictions=[]
for p in y_pred_test:
   if p==0:
      predictions.append('Low')
   elif p==1:
      predictions.append('Medium')
   else:
      predictions.append('High')

In [99]:
for i in range(10):
    print(predictions[i])

Low
Low
Low
Low
Low
Medium
Low
Medium
High
Low


Final predictions data is calculated and manipulated into a pandas dataframe

In [100]:
data= {
    'id': np.asarray(testing_data['id']),
    'irrigation_need': np.asarray(predictions)
}
final_predicted_data= pd.DataFrame(data)

In [101]:
final_predicted_data.head()

,id,irrigation_need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


Saving datafram into a CSV file

In [102]:
final_predicted_data.to_csv('submission.csv',index=False)